# Error structure

> Whether the model's errors are where it has seen few similar samples, and whether the field offsets in its errors follow location, soil type or land use

In [ ]:
import warnings
import numpy as np, pandas as pd
from scipy import stats
from scipy.linalg import LinAlgWarning
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from fastcore.test import *
from soilspecdl.all import group_folds
from ossl_fuku_transfer.data import *
from ossl_fuku_transfer.transfer import preprocessings, load_pretrained
from ossl_fuku_transfer.grouped import oof_preds
from ossl_fuku_transfer.inside import layer_features
from ossl_fuku_transfer.epo import epo_project
from ossl_fuku_transfer.baselines import field_variance
warnings.simplefilter('ignore', LinAlgWarning)

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


The cells that read the Fukushima data are marked `#| fukudata`. The outputs on this page come from a local run.

A data-centric approach trains a model on the data at hand, studies its errors, and collects more samples where it fails. It rests on a hypothesis: a model, a deep-learning model in particular, makes its largest errors on samples unlike those it trained on. If so, the samples to collect can be chosen from the spectra, before any wet chemistry.

This notebook tests the hypothesis on the Cs-137 ratio and `log1p(Kex)`, with the folds grouped by field of `20_grouped_cv` and `23_field_baselines`, so that every test field is new to the model. It compares the absolute out-of-fold error with three signals available before measuring: how far a sample lies from the training samples, how much bootstrap models disagree on it, and how extreme its prediction is. It then looks at where the errors are instead, and whether the error of each field follows its location, soil type or land use.

## Data and models

As in `23_field_baselines`: the spectra after SNV and the first derivative, and block 4 of the OSSL Kex and CEC models. The spectra are also used with the direction of the IAEA artefact of `21_iaea` projected out, because the artefact changes between measurement sessions and could make a sample look unusual for reasons unrelated to the soil:

In [ ]:
#| fukudata
fuku = load_fuku()
ids = fuku.sample_ids
src = fuku.source.reindex(ids)
ratio = pd.concat([pd.read_csv(fuku_path()/'reference'/f'cs137_ratio_{s}.csv', usecols=['ID', 'exCs137_totalCs137']) for s in ('train', 'test')])
ratio = ratio.assign(ID=ratio.ID.astype(str)).set_index('ID')['exCs137_totalCs137']
targets = {'Cs-137 ratio': ratio.reindex(ids).to_numpy(), 'Kex': np.log1p(src['soil_ex_K2O'].to_numpy(dtype=float) / mg_k2o_per_cmolc)}
fields = src['field_no'].astype(str).to_numpy()
coords = src[['Field_place(latitude)', 'Field_place(longitude)']].apply(pd.to_numeric, errors='coerce').to_numpy()
soil, land = src['Soil_type_describe'].fillna('unknown').astype(str).to_numpy(), src['fieldtype'].fillna('unknown').astype(str).to_numpy()
pipe = preprocessings()['snv_d1']
Xd = pipe.fit_transform(fuku.spectra)
B4 = np.hstack([layer_features(load_pretrained(n), Xd)['block 4'] for n in ('ossl_kex_snv_d1', 'ossl_cec_snv_d1')])
pw = shared_wns[:-1]
snv = lambda A: (A - A.mean(1, keepdims=True)) / A.std(1, keepdims=True)
pids = [i for i in sorted(set(bruker_files()) & set(spa_files()), key=int) if i not in excluded]
p_tb = (snv(load_fuku(pw, pids).spectra) - snv(load_bruker(pw, pids).spectra)).mean(0)
v = pipe.named_steps['deriv'].transform(np.r_[p_tb, p_tb[-1]][None])[0]
spaces = {'spectra': Xd, 'spectra, artefact removed': epo_project(Xd, (v / np.linalg.norm(v))[:, None]), 'block 4': B4}
ridge = (make_pipeline(StandardScaler(), Ridge()), {'ridge__alpha': np.logspace(-1, 6, 15)})
Xd.shape, B4.shape

((1389, 1675), (1389, 5120))

## Novelty and uncertainty

`novelty` is the mean distance from each test sample to its `k` nearest training samples, on whitened PCA scores fitted to the training samples, where the Euclidean distance is the Mahalanobis distance:

In [ ]:
def novelty(
    Xtr, # Training features, one row per sample
    Xte, # Test features
    k=10, # Number of neighbours
    n=20, # Number of PCA components
)->np.ndarray: # One distance per test sample
    "Mean Mahalanobis distance from each row of `Xte` to its `k` nearest rows of `Xtr`, on `n` whitened PCA components of `Xtr`"
    p = PCA(n, whiten=True, random_state=0).fit(Xtr)
    return NearestNeighbors(n_neighbors=k).fit(p.transform(Xtr)).kneighbors(p.transform(Xte))[0].mean(1)

In [ ]:
rng = np.random.default_rng(0)
Xtr_ = rng.normal(size=(200, 5))
nv_ = novelty(Xtr_, np.r_[np.zeros((1, 5)), np.full((1, 5), 4.)], n=5)
assert nv_[1] > 3 * nv_[0]

Ridge on the spectra and on block 4, with its penalty chosen by inner folds, gives the out-of-fold predictions. In each fold, the novelty of each test sample in the three feature spaces, and the uncertainty: the standard deviation of the predictions of 10 ridge models, each fitted with the chosen penalty on a bootstrap resample of the training fields:

In [ ]:
#| fukudata
oof, err, nov, unc = {}, {}, {}, {}
for t, y_all in targets.items():
    ok = np.isfinite(y_all)
    y, fo = y_all[ok], fields[ok]
    folds = group_folds(fo)
    for m, X in [('spectra', Xd[ok]), ('block 4', B4[ok])]:
        p, chosen = oof_preds(*ridge, X, y, folds, fo)
        oof[t, m], err[t, m] = p, np.abs(y - p)
        nov[t, m] = {k: np.full(len(y), np.nan) for k in spaces}
        unc[t, m] = np.full(len(y), np.nan)
        for j, f in enumerate(np.unique(folds)):
            tr, te = folds != f, folds == f
            for k, S in spaces.items(): nov[t, m][k][te] = novelty(S[ok][tr], S[ok][te])
            rng, uf, bp = np.random.default_rng(j), np.unique(fo[tr]), []
            for b in range(10):
                sel = tr & np.isin(fo, rng.choice(uf, len(uf), replace=True))
                bp.append(make_pipeline(StandardScaler(), Ridge(alpha=chosen[j]['ridge__alpha'])).fit(X[sel], y[sel]).predict(X[te]))
            unc[t, m][te] = np.std(bp, axis=0)

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. Thi

The Spearman correlation of the absolute error with each signal, and the RMSE of the fifth of the samples with the highest signal over that of the fifth with the lowest. The predicted extreme is the distance of the prediction from the mean prediction:

In [ ]:
#| fukudata
def fifth_ratio(e, s):
    q = pd.qcut(s, 5, labels=False)
    return np.sqrt((e[q == 4]**2).mean() / (e[q == 0]**2).mean())
rows = {}
for (t, m), e in err.items():
    sig = {f'novelty, {k}': s for k, s in nov[t, m].items()} | {'uncertainty': unc[t, m], 'predicted extreme': np.abs(oof[t, m] - oof[t, m].mean())}
    for k, s in sig.items(): rows[t, m, k] = {'ρ': stats.spearmanr(e, s)[0], 'RMSE ratio, top / bottom fifth': fifth_ratio(e, s)}
pd.DataFrame(rows).T.unstack(1).round(3)

ρ  ...        
                                                block 4  ... spectra
Cs-137 ratio novelty, block 4                     0.019  ...   1.058
             novelty, spectra                     0.050  ...   1.067
             novelty, spectra, artefact removed   0.050  ...   1.076
             predicted extreme                   -0.078  ...   1.013
             uncertainty                          0.094  ...   1.126
Kex          novelty, block 4                     0.132  ...   1.449
             novelty, spectra                     0.108  ...   1.264
             novelty, spectra, artefact removed   0.106  ...   1.263
             predicted extreme                    0.067  ...   1.269
             uncertainty                          0.087  ...   1.218

[10 rows x 4 columns]

## Where the errors are

For ridge on block 4: the share of the residual variance between fields (`field_variance` of `23_field_baselines`, across years), the slope of the residual on the measured value, the Spearman correlation of the absolute error with the distance of the measured value from its mean, and the standard deviation of the predictions over that of the measured values:

In [ ]:
#| fukudata
rows = {}
for t, y_all in targets.items():
    ok = np.isfinite(y_all)
    y, r = y_all[ok], y_all[ok] - oof[t, 'block 4']
    rows[t] = {'residual share between fields': field_variance(r, fields[ok])['share between'], 'slope of residual on measured value': np.polyfit(y, r, 1)[0],
               'ρ, |error| and |measured − mean|': stats.spearmanr(np.abs(r), np.abs(y - y.mean()))[0], 'sd of predictions / sd of measured': oof[t, 'block 4'].std() / y.std()}
pd.DataFrame(rows).round(3)

,Cs-137 ratio,Kex
residual share between fields,0.530,0.592
slope of residual on measured value,0.414,0.700
"ρ, |error| and |measured − mean|",0.260,0.544
sd of predictions / sd of measured,0.776,0.580


## Field residuals

The mean residual of each field, from ridge on block 4, with the field's coordinates, soil type and land use:

In [ ]:
#| fukudata
fres = {}
for t, y_all in targets.items():
    ok = np.isfinite(y_all)
    d = pd.DataFrame({'field': fields[ok], 'r': y_all[ok] - oof[t, 'block 4'], 'lat': coords[ok, 0], 'lon': coords[ok, 1], 'soil': soil[ok], 'land': land[ok]})
    fres[t] = d.groupby('field').agg(r=('r', 'mean'), lat=('lat', 'mean'), lon=('lon', 'mean'), soil=('soil', 'first'), land=('land', 'first'))
{t: len(f) for t, f in fres.items()}

{'Cs-137 ratio': 391, 'Kex': 391}

`morans_i` measures how much the residuals of nearby fields agree, with weights that average the `k` nearest fields, and tests it against random permutations of the residuals over the fields:

In [ ]:
def morans_i(
    v, # One value per location
    xy, # Coordinates of each location
    k=5, # Number of neighbours
    n_perm=999, # Number of permutations
    seed=0, # Seed for the permutations
)->tuple: # Moran's I, and its one-sided permutation p-value
    "Moran's I of `v` with row-standardised weights on the `k` nearest neighbours by `xy`"
    ix = NearestNeighbors(n_neighbors=k + 1).fit(xy).kneighbors(xy)[1][:, 1:]
    I = lambda z: (z * z[ix].mean(1)).sum() / (z**2).sum()
    z, rng = v - v.mean(), np.random.default_rng(seed)
    i0 = I(z)
    return i0, (1 + sum(I(rng.permutation(z)) >= i0 for _ in range(n_perm))) / (n_perm + 1)

In [ ]:
xy_ = np.c_[np.arange(100.), np.zeros(100)]
i_, p_ = morans_i(np.sin(np.arange(100) / 10), xy_, n_perm=199)
assert i_ > 0.8 and p_ < 0.01
i_, p_ = morans_i(np.random.default_rng(1).normal(size=100), xy_, n_perm=199)
assert abs(i_) < 0.2

Moran's I on the five nearest fields, the correlation of each field's residual with the mean residual of its nearest 1 to 20 fields, and the median distance to the nearest field in degrees:

In [ ]:
#| fukudata
rows = {}
for t, f in fres.items():
    xy, r = f[['lat', 'lon']].to_numpy(), f.r.to_numpy()
    i, p = morans_i(r, xy)
    rows[t] = {"Moran's I": i, 'p': p}
    for k in [1, 5, 10, 20]:
        ix = NearestNeighbors(n_neighbors=k + 1).fit(xy).kneighbors(xy)[1][:, 1:]
        rows[t][f'r, mean of {k} nearest'] = np.corrcoef(r, r[ix].mean(1))[0, 1]
    rows[t]['median distance to the nearest field'] = np.median(NearestNeighbors(n_neighbors=2).fit(xy).kneighbors(xy)[0][:, 1])
pd.DataFrame(rows).round(4)

,Cs-137 ratio,Kex
Moran's I,0.1345,0.0872
p,0.0010,0.0020
"r, mean of 1 nearest",0.2160,0.1782
"r, mean of 5 nearest",0.2430,0.1695
"r, mean of 10 nearest",0.2318,0.2256
"r, mean of 20 nearest",0.2041,0.1693
median distance to the nearest field,0.0075,0.0075


The share of the variance of the field residuals that soil type, land use and their combination explain (η²), with a Kruskal-Wallis test over the groups of at least 5 fields:

In [ ]:
#| fukudata
def eta2(r, g): return ((pd.Series(r).groupby(g).transform('mean') - r.mean())**2).sum() / ((r - r.mean())**2).sum()
rows = {}
for t, f in fres.items():
    for k, g in [('soil type', f.soil), ('land use', f.land), ('soil type × land use', f.soil + ', ' + f.land)]:
        g = g.to_numpy()
        rows[t, k] = {'η²': eta2(f.r.to_numpy(), g), 'p': stats.kruskal(*[f.r.to_numpy()[g == u] for u in np.unique(g) if (g == u).sum() >= 5]).pvalue}
pd.DataFrame(rows).T.round(4)

η²       p
Cs-137 ratio soil type             0.0095  0.6331
             land use              0.0068  0.0992
             soil type × land use  0.0429  0.4316
Kex          soil type             0.0225  0.2177
             land use              0.1264  0.0000
             soil type × land use  0.1802  0.0000

The mean field residual by soil type and land use, with its standard error, for the groups of at least 5 fields:

In [ ]:
#| fukudata
pd.concat({t: f.groupby(f.soil + ', ' + f.land).r.agg(fields='size', mean='mean', se=lambda s: s.std() / np.sqrt(len(s))).query('fields >= 5')
           for t, f in fres.items()}, axis=1).round(3)

Cs-137 ratio                  Kex              
                                   fields   mean     se fields   mean     se
Andosols, paddy_field                  36  0.021  0.032     36 -0.084  0.031
Andosols, upland_field                 29 -0.018  0.024     29  0.088  0.039
Forest soil, paddy_field               13  0.054  0.062     13 -0.146  0.036
Forest soil, upland_field              22 -0.065  0.049     22  0.028  0.048
Lowland soil, paddy_field             164  0.009  0.015    164 -0.027  0.012
Lowland soil, upland_field             83  0.004  0.022     83  0.057  0.019
Peat soil, paddy_field                  9  0.046  0.073      9 -0.088  0.048
Peat soil, upland_field                 9  0.076  0.080      9  0.057  0.071
Red yellow soil, paddy_field           13  0.064  0.061     13 -0.106  0.036

## Findings

The errors are not where the model has seen few similar samples. For the Cs-137 ratio, the absolute error barely correlates with novelty in any feature space (Spearman −0.03 to 0.05), with the disagreement of bootstrap models (0.04 and 0.09) or with how extreme the prediction is (−0.08 and −0.05). The fifth of the samples that each signal ranks highest has an RMSE 1.01 to 1.18 times that of the lowest fifth. For Kex the signals do slightly better: Spearman 0.03 to 0.13, and RMSE ratios of 1.22 to 1.52, highest for novelty in block 4. Removing the direction of the IAEA artefact does not change novelty's relation with the error, so measurement sessions do not hide one.

The errors are field offsets and a pull towards the mean. Between 53% (Cs-137 ratio) and 59% (Kex) of the residual variance lies between fields, so the model is wrong by a similar amount on every sample of a field. The predictions are compressed: their standard deviation is 78% and 58% of that of the measured values, the residual rises with the measured value (slopes 0.41 and 0.70), and the error grows with distance from the mean (Spearman 0.26 and 0.54). The fields with high or low values do not look unusual in their spectra.

The field offsets follow location a little. Moran's I of the field residuals on the five nearest fields is 0.13 for the Cs-137 ratio and 0.09 for Kex (p 0.001 and 0.002). A field's residual correlates with the mean residual of its nearest 1 to 20 fields at 0.20 to 0.24 for the ratio and 0.17 to 0.23 for Kex. The nearest field lies a median of 0.0075°, about 0.8 km, away.

Soil type and land use explain nothing for the Cs-137 ratio: η² is 1% to 4%, with p 0.10 to 0.63. For Kex, land use explains 13% (p < 0.001), and soil type with land use 18%. The model under-predicts upland fields (+0.03 to +0.09 on `log1p`) and over-predicts paddy fields (−0.03 to −0.15), within every soil type. Adding land use to the model's inputs takes up most of this (`23_field_baselines`).

For sampling guided by the model, these data give little to steer by. The spectra and the model do not flag the samples they will get wrong. Location helps a little for both properties, and land use for Kex. Over 80% of the variance of the field offsets is explained by none of these. What drives it is not in these data. Field management, such as potassium fertilisation, deep ploughing, topsoil removal and the history of remediation since 2011, is the obvious candidate to obtain.

All models here are ridge regressions, on the spectra or on frozen block 4. The uncertainty of an ensemble of CNNs was not tested.